## Классификация текстов с использованием предобученных языковых моделей.

В данном задании вам предстоит обратиться к задаче классификации текстов и решить ее с использованием предобученной модели BERT.

In [ ]:
import json
# do not change the code in the block below
# __________start of block__________
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import clear_output
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve

%matplotlib inline
# __________end of block__________

Обратимся к набору данных SST-2. Holdout часть данных (которая понадобится вам для посылки) доступна по ссылке ниже.

In [ ]:
# do not change the code in the block below
# __________start of block__________

!wget https://raw.githubusercontent.com/girafe-ai/ml-course/refs/heads/24f_yandex_ml_trainings/homeworks/hw04_bert_and_co/texts_holdout.json
# __________end of block__________

--2024-11-21 08:53:25--  https://raw.githubusercontent.com/girafe-ai/ml-course/refs/heads/24f_yandex_ml_trainings/homeworks/hw04_bert_and_co/texts_holdout.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 51581 (50K) [text/plain]
Saving to: ‘texts_holdout.json.3’

texts_holdout.json. 100%[===================>]  50.37K  --.-KB/s    in 0.008s  

2024-11-21 08:53:25 (5.91 MB/s) - ‘texts_holdout.json.3’ saved [51581/51581]



In [ ]:
# do not change the code in the block below
# __________start of block__________
df = pd.read_csv(
    "https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv",
    delimiter="\t",
    header=None,
)
texts_train = df[0].values[:5000]
y_train = df[1].values[:5000]
texts_test = df[0].values[5000:]
y_test = df[1].values[5000:]
with open("texts_holdout.json") as iofile:
    texts_holdout = json.load(iofile)
# __________end of block__________

Весь остальной код предстоит написать вам.

Для успешной сдачи на максимальный балл необходимо добиться хотя бы __84.5% accuracy на тестовой части выборки__.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch import nn
import torch

class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, return_tensors='pt', max_length=self.max_length, padding='max_length', truncation=True)
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'], 'label': torch.tensor(label)}

In [ ]:
from transformers import BertTokenizer, BertModel

class BERTClassifier(nn.Module):
    def __init__(self, bert_model_name, num_classes):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = outputs.pooler_output
        x = self.dropout(x)
        logits = self.fc(x)
        return logits


In [ ]:
bert_model_name = 'bert-base-uncased'
num_classes = 2
max_length = 128
batch_size = 16
num_epochs = 2
learning_rate = 2e-5

In [ ]:
tokenizer = BertTokenizer.from_pretrained(bert_model_name)

train_dataset = TextClassificationDataset(texts_train, y_train, tokenizer, max_length)
test_dataset = TextClassificationDataset(texts_test, y_test, tokenizer, max_length)
holdout_dataset = TextClassificationDataset(texts_holdout, [0] * len(texts_holdout), tokenizer, max_length)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)
holdout_dataloader = DataLoader(holdout_dataset, batch_size=batch_size)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BERTClassifier(bert_model_name, num_classes).to(device)

In [ ]:
from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
total_steps = len(train_dataloader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [ ]:
from tqdm.auto import tqdm

def train(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    i = 0
    for batch in tqdm(dataloader, desc='Batches'):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()
        i += 1

        if i % 100 == 0:
          print(total_loss / i)
          total_loss = 0




In [ ]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate(model, dataloader, device):
    model.eval()
    pred = []
    actual_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Batches'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
            pred.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())

    return accuracy_score(actual_labels, pred)


In [ ]:
def predict(model, dataloader, device):
    model.eval()
    probs = []
    softmax = nn.Softmax(dim=1)

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Batches'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = softmax(model(input_ids=input_ids, attention_mask=attention_mask))[:, 1]
            probs.extend(outputs.cpu().tolist())

    return probs

In [ ]:
for epoch in tqdm(range(num_epochs), desc='Epochs'):
    train(model, train_dataloader, optimizer, scheduler, device)
    accuracy = evaluate(model, test_dataloader, device)
    print(f"Test Accuracy: {accuracy:.4f}")


Epochs:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

0.4585270604491234
0.1482684637606144
0.09636885322630405


Batches:   0%|          | 0/120 [00:00<?, ?it/s]

Test Accuracy: 0.9094


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

0.14451552711427212
0.05933458441868424
0.04380117472416411


Batches:   0%|          | 0/120 [00:00<?, ?it/s]

Test Accuracy: 0.9099


In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

out_dict = {
    "train": predict(model, train_dataloader, device),
    "test": predict(model, test_dataloader, device),
    "holdout": predict(model, holdout_dataloader, device),
}


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Batches:   0%|          | 0/120 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

[0.054883603006601334, 0.98988276720047, 0.9602424502372742, 0.006165020167827606, 0.007518771104514599, 0.44191262125968933, 0.9874502420425415, 0.003876624396070838, 0.992418646812439, 0.9958065748214722, 0.15200944244861603, 0.0037665781565010548, 0.1622128188610077, 0.9829035401344299, 0.011569050140678883, 0.9945340156555176, 0.22652871906757355, 0.9963952898979187, 0.060748714953660965, 0.05275280028581619, 0.005560020916163921, 0.0061745173297822475, 0.9963952898979187, 0.9890196919441223, 0.9963000416755676, 0.9919387698173523, 0.9738044738769531, 0.9952669143676758, 0.021670976653695107, 0.041426125913858414, 0.8540341854095459, 0.6282339096069336, 0.0057975659146904945, 0.9954383969306946, 0.9962653517723083, 0.9958677291870117, 0.003799332305788994, 0.035082604736089706, 0.9904693961143494, 0.9973335266113281, 0.9838061332702637, 0.005408338271081448, 0.8518860340118408, 0.017223607748746872, 0.04744512215256691, 0.011152367107570171, 0.946591317653656, 0.9602424502372742, 0

In [ ]:
print(type(out_dict['holdout'][0]))

<class 'float'>


#### Сдача взадания в контест
Сохраните в словарь `out_dict` вероятности принадлежности к первому (положительному) классу

Несколько `assert`'ов для проверки вашей посылки:

In [ ]:
assert isinstance(out_dict["train"], list), "Object must be a list of floats"
assert isinstance(out_dict["train"][0], float), "Object must be a list of floats"
assert (
    len(out_dict["train"]) == 5000
), "The predicted probas list length does not match the train set size"

assert isinstance(out_dict["test"], list), "Object must be a list of floats"
assert isinstance(out_dict["test"][0], float), "Object must be a list of floats"
assert (
    len(out_dict["test"]) == 1920
), "The predicted probas list length does not match the test set size"

assert isinstance(out_dict["holdout"], list), "Object must be a list of floats"
assert isinstance(out_dict["holdout"][0], float), "Object must be a list of floats"
assert (
    len(out_dict["holdout"]) == 500
), "The predicted probas list length does not match the holdout set size"

Запустите код ниже для генерации посылки.

In [ ]:
# do not change the code in the block below
# __________start of block__________
FILENAME = "submission_dict_hw_text_classification_with_bert.json"

with open(FILENAME, "w") as iofile:
    json.dump(out_dict, iofile)
print(f"File saved to `{FILENAME}`")
# __________end of block__________

File saved to `submission_dict_hw_text_classification_with_bert.json`


На этом задание завершено. Поздравляем!